# 02 — Prompt Engineering and Structured Outputs

**First Finance - Arnaud Demes**

Turn NVIDIA filing evidence into a typed analyst brief that the Financial Analyst Copilot can validate, test, and reuse.

## Learning objectives

By the end of this notebook, you can:

- separate stable instructions, trusted inputs, source data, and output constraints;
- explain why valid JSON can still be financially invalid;
- define a typed financial response with Pydantic;
- generate the same structured output with Ollama or OpenAI; and
- apply deterministic checks after model generation.

## Before you start

**Estimated time:** 20 minutes after the 10-minute concept deck.

Run this notebook from the course environment. In class, choose one provider:

```bash
FINAI_MODEL_PROVIDER=ollama FINAI_CHAT_MODEL=qwen3:8b uv run --extra ai jupyter lab
```

```bash
FINAI_MODEL_PROVIDER=openai FINAI_CHAT_MODEL=gpt-5-mini uv run --extra ai jupyter lab
```

For OpenAI, define `OPENAI_API_KEY` in the shell before starting Jupyter. Never paste an API key into a notebook. Automated tests set `FINAI_LIVE_MODE=0` and use a deterministic fixture instead of contacting a provider.

## Where this fits

Lesson 1 established one provider-neutral model boundary. Lesson 2 changes the output boundary from free-form text to a validated financial object.

```text
NVIDIA evidence → versioned prompt → structured model → AnalystBrief → deterministic checks
```

The same `AnalystBrief` becomes the capstone contract for the later user interface, retrieval workflow, tools, agent, and evaluations.

## A prompt is an application interface

An analyst request such as `Give me the main NVIDIA results and risks` omits the period, authorised evidence, treatment of uncertainty, and required output. A production prompt makes four boundaries explicit:

| Boundary | Responsibility |
|---|---|
| System instructions | Stable role, evidence policy, prohibited behaviour |
| Trusted application inputs | Company and reporting period selected by code |
| Source document | Untrusted data that may be incomplete or adversarial |
| Output contract | Fields, types, allowed categories, and required evidence |

Prompt text guides generation. The schema and Python validators decide what the application accepts.

In [ ]:
from __future__ import annotations

import json
import os
from pprint import pprint

from pydantic import ValidationError

from finai_academy.capstone import (
    AnalystBrief,
    AnalystBriefService,
    EvidenceType,
    create_structured_model,
)
from finai_academy.lesson_support import RecordedStructuredModel
from finai_academy.settings import Settings

## Reuse the Lesson 1 NVIDIA evidence

Keeping the company and evidence stable isolates the new engineering concept. The source is NVIDIA's fiscal 2026 Form 10-K filed with the U.S. Securities and Exchange Commission: [SEC filing](https://www.sec.gov/Archives/edgar/data/1045810/000104581026000021/nvda-20260125.htm).

Each fact retains a stable identifier for discussion, but the structured brief will also preserve short source excerpts.

In [ ]:
source_document = """
[F1] NVIDIA fiscal 2026 revenue was $215.9 billion, up 65% year on year.
[F2] Data Center revenue was $193.7 billion, up 68% year on year.
[F3] Gaming revenue was $16.0 billion, up 41% year on year.
[F4] Gross margin decreased; the filing says it was also affected by a $4.5 billion H20 excess-inventory and purchase-obligation charge.
""".strip()

print(source_document)

## Failure lab

The candidate below is syntactically valid JSON. It is still unacceptable because a `reported_fact` has no source excerpt. This is the distinction between **parseable output** and an **accepted financial object**.

> **Before running the cell:** predict whether `json.loads` or Pydantic will reject the candidate.

In [ ]:
invalid_candidate = {
    "company": "NVIDIA",
    "reporting_period": "fiscal 2026",
    "executive_summary": "Revenue increased.",
    "findings": [
        {
            "statement": "Revenue increased 65%.",
            "category": "key_result",
            "evidence_type": "reported_fact",
            "source_excerpt": None,
        }
    ],
    "open_questions": [],
    "caveats": [],
}

candidate_json = json.dumps(invalid_candidate)
json.loads(candidate_json)
print("PASS — JSON syntax is valid")

try:
    AnalystBrief.model_validate_json(candidate_json)
except ValidationError as error:
    print("Validation caught the unsupported candidate")
    print(error.errors()[0]["msg"])
else:
    raise AssertionError("The invalid financial candidate was unexpectedly accepted.")

## The `AnalystBrief` contract

The capstone schema represents a financial product, not a generic JSON response:

- `AnalystBrief` fixes the company, reporting period, summary, findings, questions, and caveats;
- `AnalystFinding.category` is limited to key result, catalyst, or risk;
- `evidence_type` separates reported facts, calculations, management claims, external facts, and interpretations;
- reported facts and management claims require a short source excerpt;
- interpretations require a rationale; and
- unexpected fields are rejected.

These constraints remain useful even when the model provider changes.

In [ ]:
schema = AnalystBrief.model_json_schema()
print("Top-level fields:", list(schema["properties"]))
print("Extra fields allowed:", AnalystBrief.model_config.get("extra") != "forbid")

### Schema-bound generation

OpenAI Structured Outputs are designed to make responses adhere to a supplied JSON Schema. The SDK can derive that schema from Pydantic, while application code still handles refusals, provider errors, and business validation. See the [official OpenAI Structured Outputs guide](https://developers.openai.com/api/docs/guides/structured-outputs).

Our capstone exposes the narrower `StructuredModel` protocol. The LangChain adapters bind the same Pydantic class for Ollama and OpenAI, so the notebook is not coupled to either SDK.

## Select the provider once

In a normal student run, `FINAI_LIVE_MODE` defaults to `1`. The configured Ollama or OpenAI adapter is constructed. The automated course suite sets it to `0` and substitutes a recorded implementation of the same structured-model protocol.

> **Expected result:** class execution prints `live / ollama`, `live / openai`, or `offline fixture`. It must never display a secret.

In [ ]:
settings = Settings.from_environment()
live_mode = os.getenv("FINAI_LIVE_MODE", "1") == "1"
structured_model = (
    create_structured_model(settings) if live_mode else RecordedStructuredModel()
)

print("Execution mode:", f"live / {settings.provider}" if live_mode else "offline fixture")
print("Configured model:", settings.chat_model)

## Generate through the application service

`AnalystBriefService` owns the stable system prompt and prompt version. It wraps the source in explicit delimiters and tells the model to treat document contents as untrusted data.

The application—not the model—owns the selected company and period. After generation, the service restores these trusted inputs even if the model returns different values.

In [ ]:
service = AnalystBriefService(structured_model)
brief = service.generate(
    company="NVIDIA",
    reporting_period="fiscal 2026",
    source_text=source_document,
)

pprint(brief.model_dump(mode="json"), sort_dicts=False)

## Three validation layers

1. **Syntax:** can Python parse the response?
2. **Schema:** do required fields, types, enums, and forbidden extras match?
3. **Finance semantics:** is evidence attached where required, are interpretations identified, and is uncertainty visible?

Structured generation improves the second layer. It does not remove the need for the third. The checks below are deliberately deterministic and inspect the accepted object rather than asking another model to grade it.

## Verification

The verification cell checks that the result is typed, preserves trusted inputs, contains at least one finding, satisfies every evidence rule, and states at least one caveat. A live model failure is evidence to inspect, not a reason to bypass the contract.

> **Target result:** every line prints `PASS`, followed by `PASS — structured financial brief verified`.

In [ ]:
evidence_rules_hold = all(
    (
        finding.evidence_type
        not in {EvidenceType.REPORTED_FACT, EvidenceType.MANAGEMENT_CLAIM}
        or bool((finding.source_excerpt or "").strip())
    )
    and (
        finding.evidence_type != EvidenceType.INTERPRETATION
        or bool((finding.rationale or "").strip())
    )
    for finding in brief.findings
)
checks = {
    "typed AnalystBrief": isinstance(brief, AnalystBrief),
    "trusted company": brief.company == "NVIDIA",
    "trusted reporting period": brief.reporting_period == "fiscal 2026",
    "at least one finding": bool(brief.findings),
    "evidence requirements": evidence_rules_hold,
    "explicit caveat": bool(brief.caveats),
}

for criterion, passed in checks.items():
    print(f"{'PASS' if passed else 'REVIEW':6} {criterion}")
assert all(checks.values()), "Review the structured brief against the visible criteria."
print("PASS — structured financial brief verified")

## Failure handling in a real application

A production boundary distinguishes at least four outcomes:

- provider or network error — retry only when the error is transient;
- refusal — preserve and display the refusal explicitly;
- schema validation error — log the prompt version and validation details;
- semantic review failure — return a bounded failure or send the result to human review.

Do not silently coerce an unsupported financial claim into a valid object. A retry can improve formatting; it cannot manufacture missing evidence.

## Challenge

Add `confidence_reason: str` to `AnalystFinding`. Require a concise explanation of why the evidence is sufficient or limited. Do **not** add an ungrounded numeric confidence score.

Use the engineering sequence:

1. write a failing validation test;
2. change the Pydantic contract;
3. update the offline fixture and prompt;
4. rerun the focused tests and this notebook; and
5. decide whether the field belongs in every later capstone view.

Optional comparison: run once with Ollama and once with OpenAI, then compare validation success, latency, and the usefulness of the caveats—not just writing style.

## Troubleshooting

| Symptom | Likely cause | Action |
|---|---|---|
| `ModuleNotFoundError` | Jupyter was not started in the course environment | Restart with `uv run --extra ai jupyter lab` from the repository root |
| Ollama connection error | Ollama is stopped or the base URL is wrong | Start Ollama and run `scripts/setup_check.py --provider ollama` |
| OpenAI authentication error | `OPENAI_API_KEY` is absent from the Jupyter process | Export it in the shell, restart Jupyter, and never paste it here |
| Structured validation error | The model output violates the application contract | Read the exact Pydantic error before changing the prompt or schema |
| No caveat returned | The response is incomplete for this lesson's contract | Strengthen the uncertainty instruction or route to review |
| Output says `offline fixture` | `FINAI_LIVE_MODE=0` is set | Remove the variable and restart the kernel for a live run |

## Capstone integration

The Financial Analyst Copilot now owns a reusable vertical slice:

1. `AnalystBrief` and `AnalystFinding` define the financial object;
2. `AnalystBriefService` owns the versioned evidence-disciplined prompt;
3. `StructuredModel` keeps Ollama and OpenAI behind one boundary;
4. Pydantic rejects unsupported facts and unexplained interpretations; and
5. deterministic post-generation checks decide whether the brief is accepted.

Lesson 3 will expand the source from a compact evidence card to a complete financial document and examine when full-context prompting is sufficient.

## Recap

- Prompts specify the task; schemas specify the accepted object.
- Valid JSON can still violate the financial evidence contract.
- Pydantic provides types, enums, forbidden extras, and domain validators.
- Trusted application inputs should not be delegated to the model.
- Ollama and OpenAI can produce the same typed capstone artifact.
- Structured generation reduces formatting failures but does not guarantee financial correctness.
- Deterministic checks and explicit failure paths remain essential.